# Create embeddings in the database (Azure OpenAI)

This notebook has Azure SQL call your Azure OpenAI embedding model itself, so turning text into a vector is plain T-SQL:

1. **Register the model** with `CREATE EXTERNAL MODEL`, once.
2. **Embed the 12 articles** with `AI_GENERATE_EMBEDDINGS`, and store the vectors next to them.
3. **Ask a question in plain English.** The database embeds the question too, ranks the articles by meaning, and adds business data, in one statement.

**Before you start:** run `npm --prefix app/ai-service run credential`, as in walkthrough step 6. It stores your Azure OpenAI key in the database, where only the database can use it. The key never appears in this notebook.

Run the cells one at a time, top to bottom, with the **Run** button beside each. The first one asks for a connection: choose **CaseDesk (local)**.

In [ ]:
%%use CaseDesk

`%%use CaseDesk` above is a notebook command, not T-SQL: it connects and switches to the `CaseDesk` database.

## 1. Register the model

`CREATE EXTERNAL MODEL` tells the database where your embedding model lives and which credential holds its key. It asks the model for 768 numbers per text, to fit the `VECTOR(768)` column.

**Edit first:** replace `<your-resource>` in both places with the resource name the `credential` command printed. If the command also printed a deployment name, replace `text-embedding-3-small` in both places too.

The cell drops and re-creates the model, so you can run it again. Creating the model doesn't call it yet; the check after it catches a typo before the first call does.

In [ ]:
IF EXISTS (SELECT 1 FROM sys.external_models WHERE name = N'casedesk_embeddings')
    DROP EXTERNAL MODEL casedesk_embeddings;

CREATE EXTERNAL MODEL casedesk_embeddings
WITH (
    LOCATION   = 'https://<your-resource>.openai.azure.com/openai/deployments/text-embedding-3-small/embeddings?api-version=2024-10-21',
    API_FORMAT = 'Azure OpenAI',
    MODEL_TYPE = EMBEDDINGS,
    MODEL      = 'text-embedding-3-small',
    CREDENTIAL = [https://<your-resource>.openai.azure.com/],
    PARAMETERS = '{"dimensions":768}'
);

**Check:** one row, with `status` showing **ready**. The model's address must start with the credential's name; that's how the database knows which key to send.

In [ ]:
SELECT m.name,
       m.model,
       m.parameters,
       c.name AS credential,
       CASE WHEN c.name IS NULL THEN N'no credential: run the credential command'
            WHEN LEFT(m.location, LEN(c.name)) = c.name THEN N'ready'
            ELSE N'address and credential differ: check <your-resource>' END AS status
FROM sys.external_models AS m
LEFT JOIN sys.database_scoped_credentials AS c ON c.credential_id = m.credential_id
WHERE m.name = N'casedesk_embeddings';

## 2. Embed the articles

`AI_GENERATE_EMBEDDINGS` sends text to the model and returns its vector, right inside a query. This cell embeds every article that has no current embedding: one call per article, a few seconds for all 12. Run it again later, and it only embeds articles that changed.

Each row gets the same label the AI service uses, `azure-openai:<deployment>:768`, and a hash of the article's text, so the app uses these vectors as they are.

**Expect:** 12 for both counts.

**Msg 31622?** Some preview builds of the local container can't call out yet. Run `npm --prefix app/ai-service run embed` in the command terminal instead: the AI service stores the same vectors with the same label. Then skip section 3; step 10 runs it in Azure.

In [ ]:
SET XACT_ABORT ON;

DECLARE @model nvarchar(100) =
    (SELECT N'azure-openai:' + model + N':768' FROM sys.external_models WHERE name = N'casedesk_embeddings');

BEGIN TRANSACTION;

-- Remove embeddings from another model, or made before the article last changed.
DELETE e
FROM dbo.kb_article_embeddings AS e
JOIN dbo.kb_articles AS a ON a.id = e.article_id
WHERE e.embedding_model <> @model
   OR e.content_hash <> HASHBYTES('SHA2_256', CONCAT(a.title, NCHAR(10), a.content));

INSERT INTO dbo.kb_article_embeddings (article_id, embedding_model, content_hash, embedding)
SELECT a.id,
       @model,
       HASHBYTES('SHA2_256', t.article_text),
       AI_GENERATE_EMBEDDINGS(t.article_text USE MODEL casedesk_embeddings)
FROM dbo.kb_articles AS a
CROSS APPLY (SELECT CONCAT(a.title, NCHAR(10), a.content) AS article_text) AS t
WHERE NOT EXISTS (SELECT 1 FROM dbo.kb_article_embeddings AS e WHERE e.article_id = a.id);

COMMIT TRANSACTION;

SELECT COUNT(*) AS embedded_articles,
       (SELECT COUNT(*) FROM dbo.kb_articles) AS total_articles
FROM dbo.kb_article_embeddings
WHERE embedding_model = @model;

## 3. Ask a question in plain English

This is the whole search in one statement:

1. `AI_GENERATE_EMBEDDINGS` turns the question into a vector with the same model as the articles, so the two can be compared.
2. `VECTOR_DISTANCE('cosine', ...)` ranks the articles by meaning: a smaller distance is closer. It compares only vectors with the same label.
3. The 3 closest articles get business data: the number of open cases in their product area.

No app code is involved, so a stored procedure, a report, or Data API builder could run the same search. Change the question and run it again; each run makes one call to the model.

**Expect:** three articles, closest first. For the sample question, **KB-3, Data sync is delayed or stuck in Pending**, comes first.

In [ ]:
DECLARE @question nvarchar(2000) = N'Field updates from the mobile app have been stuck in Pending since this morning.';

DECLARE @model nvarchar(100) =
    (SELECT N'azure-openai:' + model + N':768' FROM sys.external_models WHERE name = N'casedesk_embeddings');

DECLARE @query_vector vector(768) = AI_GENERATE_EMBEDDINGS(@question USE MODEL casedesk_embeddings);

-- Rank by meaning first, then enrich only the closest articles with business data.
WITH nearest AS (
    SELECT TOP (3)
           e.article_id,
           VECTOR_DISTANCE('cosine', e.embedding, @query_vector) AS cosine_distance
    FROM dbo.kb_article_embeddings AS e
    WHERE e.embedding_model = @model
    ORDER BY cosine_distance
)
SELECT a.id,
       a.title,
       a.product_area,
       n.cosine_distance,
       (SELECT COUNT(*)
          FROM dbo.support_cases AS c
         WHERE c.product_area = a.product_area
           AND c.status <> N'Resolved') AS open_cases
FROM nearest AS n
JOIN dbo.kb_articles AS a ON a.id = n.article_id
ORDER BY n.cosine_distance;